## get_action 核心方法

1. 历史动作处理(action_history不空):
   1. 提取最后一步动作 `a_{t-1}`:
      - 如果是`calculate`动作，使用`eval`计算并添加到`action_history[-1]`中 -> 在`progress_summary_prompt`和`get_user_msgs_nonchat`中使用
      - 如果是`take_note`动作，则包含在`note_contents`中 -> 在`get_user_msgs_nonchat`中使用
   2. 使用`progress_summary_prompt`,对应论文中$m_t = \pi_\text{sum}(m_{t-1}, o_t, a_{t-1}, G)$ 实际上$G$在`get_user_msgs_nonchat`中使用 
2. 构建系统消息`get_system_msgs`
   - 基本指令
     - 指令Instructions："你是一个UI助手，..."
     - 响应格式response format："<think>...</think><action>...</action>"
     - 通用提示General Tips
     - 答案内容上的要求
   - 动作空间描述`action_set.describe()`+示例
   - 下一步的动作指令："基于以上信息，决定你的下一步动作，..."
3. 构建用户消息`get_user_msgs_nonchat`
   - 目标: `obs['goal_object']`
   - 标签页信息: `obs['open_pages_urls', 'open_pages_titles', 'active_page_index']`
   - 可访问性树、html、截图[可选]: `obs['axtree_txt', 'pruned_html', 'screenshot_som']`
   - 操作历史: `action_history`
   - 最后一次操作的错误信息: `last_action_error`
   - 笔记内容: `note_contents`
   - 总结内容: `progress_summary_content`
4. 合并系统消息和用户消息，形成完整提示`full_prompt_txt`
5. 调用OpenAI API获取响应，解析获取`action`，如果需要，特殊处理`action`
6. 将新动作添加到`action_history`，返回`action`和`full_prompt_txt`

## progress_summary 方法

```py
def progress_summary(self, obs: dict) -> str:  # 论文中 m_t = π_sum(m_{t-1}, o_t(D_t, V_t), a_{t-1}, G)
    messages = [
        {"role": "system", "content": PROGRESS_SUMMARY_PROMPT},  # 总结提示词
    ]
    axtree_txt = obs.get("axtree_txt", "")
    # 限制长度以避免无关的冗长内容占用上下文
    if isinstance(axtree_txt, str) and len(axtree_txt) > 4000:
        axtree_txt = axtree_txt[:4000] + "\n...[truncated]..."
    user_contents = [
        {"type": "text", "text": f"# goal\n{obs['goal']}\n"},  # NT: G: 任务目标
        {"type": "text", "text": f"# axtree_txt\n{axtree_txt}\n"},  # NT: D_t: 可访问性树
        {"type": "text", "text": f"# screenshot\n"},
        {"type": "image_url", "image_url": {
            "url": image_to_jpg_base64_url(obs["screenshot_som"]),  # NT: V_t: 叠加了DOM元素标记的截图
            "detail": "auto",
        }},
        {"type": "text", "text": f"# action_history\n{'\n'.join(self.action_history)}\n"},  # NT: a,包含上一步动作 a_{t-1}
        {"type": "text", "text": f"# previous_summary\n{self.progress_summary_content}\n"},  # NT: m_{t-1}: 上一步总结
    ]

    messages.append({"role": "user", "content": user_contents})
    response = self.openai_client.chat.completions.create(
        model=self.model_name,  # Qwen2.5-VL-72B-Instruct
        messages=messages
    )  # NT: 对应论文4.2.1 1.基于视觉语言模型的判别器
    self.progress_summary_content = response.choices[0].message.content.strip()
    return self.progress_summary_content
```